# 🏨 Audit & Preprocessing Dataset: Prediksi Risiko Pembatalan Reservasi Hotel
### **SDG 8: Pekerjaan Layak & Pertumbuhan Ekonomi - Platform Analitik Prediktif (MVP)**

Notebook ini mendokumentasikan proses audit kualitas data, pembersihan anomali, eliminasi *data leakage*, rekayasa fitur (*feature engineering*), serta pemisahan dataset (*stratified train-val-test split*) untuk memastikan model machine learning (CatBoost, Random Forest, Logistic Regression) dilatih secara adil, objektif, dan siap produksi.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

# Set style visualisasi
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 11

RAW_DATA_PATH = Path('../../data/raw/hotel_booking.csv')
PROCESSED_DIR = Path('../../data/processed')

print("Environment & Library siap.")

--- 
## 1. Audit Dataset Mentah (`hotel_booking.csv`)
Memeriksa struktur awal, ukuran baris/kolom, dan missing value pada dataset mentah.

In [ ]:
raw_df = pd.read_csv(RAW_DATA_PATH)
print(f"Ukuran Dataset Mentah: {raw_df.shape[0]:,} baris x {raw_df.shape[1]} kolom")
raw_df.head(3)

In [ ]:
# Analisis Missing Values
missing_summary = raw_df.isnull().sum()
missing_summary = missing_summary[missing_summary > 0].sort_values(ascending=False)
missing_pct = (missing_summary / len(raw_df)) * 100

missing_df = pd.DataFrame({'Missing Rows': missing_summary, 'Percentage (%)': missing_pct.round(2)})
print("=== Missing Value Audit pada Data Mentah ===")
missing_df

--- 
## 2. Pemeriksaan Anomali Data
- Nilai ADR negatif (`adr < 0`)
- Reservasi tanpa tamu sama sekali (`adults == 0 & children == 0 & babies == 0`)

In [ ]:
neg_adr = raw_df[raw_df['adr'] < 0]
print(f"Jumlah baris dengan ADR < 0: {len(neg_adr)}")
display(neg_adr[['hotel', 'adr', 'customer_type', 'is_canceled']])

zero_guests = raw_df[(raw_df['adults'] == 0) & (raw_df['children'] == 0) & (raw_df['babies'] == 0)]
print(f"Jumlah reservasi tanpa tamu (adults=0, children=0, babies=0): {len(zero_guests)}")

--- 
## 3. Strict Zero-Leakage & Feature Engineering Pipeline
Menjalankan modul `data_preprocessing.py` untuk mengeliminasi kolom leakage (`reservation_status`, `reservation_status_date`), kolom identitas artifisial (`name`, `email`, `phone-number`, `credit_card`), serta kolom anonim (`company`, `agent`).

In [ ]:
from ml.experiments.data_preprocessing import (
    LEAKAGE_AND_EXCLUDED_COLUMNS,
    clean_raw_data,
    drop_unusable_columns,
    engineer_features,
)

# 1. Clean anomalies
cleaned = clean_raw_data(raw_df)

# 2. Drop leakage
filtered = drop_unusable_columns(cleaned)

# 3. Engineer features
processed = engineer_features(filtered)

print(f"Dataset setelah preprocessing: {processed.shape[0]:,} baris x {processed.shape[1]} kolom")
print("Verifikasi kolom leakage terhapus:")
for col in LEAKAGE_AND_EXCLUDED_COLUMNS:
    assert col not in processed.columns, f"Leakage detected: {col}"
print("✅ SEMUA KOLOM LEAKAGE DAN ARTIFISIAL BERHASIL DIBUANG!")

--- 
## 4. Analisis Distribusi Kelas Target (`is_canceled`)
Melihat perbandingan pemesanan yang dibatalkan (Class 1) vs tidak dibatalkan (Class 0).

In [ ]:
counts = processed['is_canceled'].value_counts()
percentages = processed['is_canceled'].value_counts(normalize=True) * 100

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(['Not Canceled (0)', 'Canceled (1)'], counts.values, color=['#22c55e', '#ef4444'], width=0.5)
for bar in bars:
    yval = bar.get_height()
    pct = (yval / len(processed)) * 100
    ax.text(bar.get_x() + bar.get_width()/2, yval + 1000, f"{yval:,}\n({pct:.1f}%)", ha='center', fontweight='bold')

ax.set_title('Distribusi Kelas Target is_canceled')
ax.set_ylabel('Jumlah Reservasi')
ax.set_ylim(0, max(counts.values) * 1.15)
plt.tight_layout()
plt.show()

--- 
## 5. Eksplorasi Fitur Hasil Rekayasa (*Engineered Features*)

In [ ]:
# Analisis cancellation rate berdasarkan lead time bins
processed['lead_time_bin'] = pd.qcut(processed['lead_time'], q=5, duplicates='drop')
lead_cancel_rate = processed.groupby('lead_time_bin')['is_canceled'].mean() * 100

fig, ax = plt.subplots(figsize=(8, 4))
lead_cancel_rate.plot(kind='bar', ax=ax, color='#3b82f6')
ax.set_title('Tingkat Pembatalan Berdasarkan Lead Time (Quintiles)')
ax.set_ylabel('Tingkat Pembatalan (%)')
ax.set_xlabel('Rentang Lead Time (Hari)')
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

--- 
## 6. Verifikasi Stratified Train, Validation, dan Test Splits

In [ ]:
with open(PROCESSED_DIR / 'feature_metadata.json') as f:
    metadata = json.load(f)

print("=== Metadata Split Info ===")
print(json.dumps(metadata['split_info'], indent=2))

print(f"\nJumlah Fitur Numerik: {len(metadata['numerical_features'])}")
print(f"Jumlah Fitur Kategorikal: {len(metadata['categorical_features'])}")
print("\n✅ Pipeline data siap untuk tahap benchmarking model!")